In [1]:
import json
import pandas as pd
from datetime import datetime
from IPython.display import display, Markdown
from os import listdir as ls
import pycountry

from emu_renewal.inputs import DATA_PATH, END_VACC_THRESHOLD, TEXT_DATE_FORMAT, get_country_vacc_data, get_indicator_series_from_who_data
from emu_renewal.run import find_run_end_time
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import is_mostly_zeros, find_null_data, find_neg_data, has_repeats, has_outlier

In [2]:
g_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
fb_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
either_mob_avail = list(set(g_avail + fb_avail))

summary = pd.DataFrame(index=either_mob_avail)
add_bool_row_to_table(summary, g_avail, "Google avail.")
add_bool_row_to_table(summary, fb_avail, "FB avail.")
txt = "To select countries for inclusion in our analysis, we first identified all countries " \
    "for which either Google or Facebook mobility data were available. "

In [3]:
# Gather data
case_data = {}
death_data = {}
for c in either_mob_avail:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, END_VACC_THRESHOLD, c)

    data = get_indicator_series_from_who_data("New_cases", c)
    date_filter = (data.index < end_time)
    case_data[c] = data[date_filter]

    data = get_indicator_series_from_who_data("New_deaths", c)
    date_filter = (data.index < end_time)
    death_data[c] = data[date_filter]

txt += "Next, we considered the quality of the data for our two main WHO indicators " \
    'which we required for inclusion in the analysis: "New_cases" and "New_deaths" ' \
    "from the start of data availability through to the end time of the analysis for each country. "

In [4]:
# Select countries based on data characteristics
n_repeats = 6
outlier_threshold = 2.0

no_cases = find_null_data(case_data)
no_deaths = find_null_data(death_data)
mostly_zeroes = [c for c, d in case_data.items() if is_mostly_zeros(d)]
add_bool_row_to_table(summary, no_cases, "No cases")
add_bool_row_to_table(summary, no_deaths, "No deaths")
txt += "\n\nUsing these data, we excluded any countries for which either case or death data was absent " \
    "and also excluded any countries for which the majority of case count estimates were zero. "

neg_cases = find_neg_data(case_data)
neg_deaths = find_neg_data(death_data)
add_bool_row_to_table(summary, no_cases, "Neg. cases")
add_bool_row_to_table(summary, no_cases, "Neg. deaths")
txt += "We also excluded any countries for which any negative values were present in the available data. "

repeat_threshold = 1e-10
case_outliers = [c for c, d in case_data.items() if has_outlier(d, outlier_threshold)]
death_outliers = [c for c, d in death_data.items() if has_outlier(d, outlier_threshold)]
add_bool_row_to_table(summary, case_outliers, "Case outliers")
add_bool_row_to_table(summary, death_outliers, "Death outliers")
txt += "We further excluded any countries for which single marked outliers were present, " \
    f"which we defined as a single value that was more than {round(outlier_threshold)} " \
    "times greater than the next highest estimate available during the analysis period. "

start_time = datetime(2020, 4, 1)
case_repeats = [c for c, d in case_data.items() if has_repeats(d[d.index > start_time], n_repeats, repeat_threshold)]
death_repeats = [c for c, d in death_data.items() if has_repeats(d[d.index > start_time], n_repeats, repeat_threshold)]
add_bool_row_to_table(summary, case_repeats, "Rep-eated cases")
add_bool_row_to_table(summary, death_repeats, "Rep-eated deaths")
txt += "Last we excluded any countries for which several repeated values were present, " \
    "or the change from each value to the subsequent reported was identical for several values. " \
    f"We set the threshold number of repeated values or repeated changes for exclusion at {n_repeats} consecutive repeats " \
    f"and required that these repeated values occur after {start_time.strftime(TEXT_DATE_FORMAT)} because " \
    "these repeated values tended to be small and less significant for calibration prior to this date."
display(Markdown(txt))

To select countries for inclusion in our analysis, we first identified all countries for which either Google or Facebook mobility data were available. Next, we considered the quality of the data for our two main WHO indicators which we required for inclusion in the analysis: "New_cases" and "New_deaths" from the start of data availability through to the end time of the analysis for each country. 

Using these data, we excluded any countries for which either case or death data was absent and also excluded any countries for which the majority of case count estimates were zero. We also excluded any countries for which any negative values were present in the available data. We further excluded any countries for which single marked outliers were present, which we defined as a single value that was more than 2 times greater than the next highest estimate available during the analysis period. Last we excluded any countries for which several repeated values were present, or the change from each value to the subsequent reported was identical for several values. We set the threshold number of repeated values or repeated changes for exclusion at 6 consecutive repeats and required that these repeated values occur after 01/04/2020 because these repeated values tended to be small and less significant for calibration prior to this date.

In [5]:
excluded = set(no_cases + no_deaths + neg_cases + neg_deaths + case_repeats + death_repeats + case_outliers + death_outliers + mostly_zeroes)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Inc-luded")
included.append("AUS")

In [6]:
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.to_markdown()))

|                                       | Google avail.   | FB avail.   | No cases   | No deaths   | Neg. cases   | Neg. deaths   | Case outliers   | Death outliers   | Rep-eated cases   | Rep-eated deaths   | Inc-luded   |
|:--------------------------------------|:----------------|:------------|:-----------|:------------|:-------------|:--------------|:----------------|:-----------------|:------------------|:-------------------|:------------|
| Iceland                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Algeria                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Guadeloupe                            | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Namibia                               | Yes             | Yes         | Yes        | Yes         | Yes          | Yes           | No              | No               | No                | No                 | No          |
| Saudi Arabia                          | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Benin                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Cameroon                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Congo                                 | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Afghanistan                           | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Senegal                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Tunisia                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Burundi                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Estonia                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Korea, Republic of                    | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Portugal                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| French Guiana                         | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Sao Tome and Principe                 | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Ethiopia                              | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Croatia                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| South Africa                          | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Austria                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Bolivia, Plurinational State of       | Yes             | Yes         | No         | No          | No           | No            | No              | Yes              | No                | No                 | No          |
| Thailand                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| El Salvador                           | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Malaysia                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Ghana                                 | Yes             | No          | No         | No          | No           | No            | No              | Yes              | No                | No                 | No          |
| Bangladesh                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Paraguay                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Switzerland                           | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Ecuador                               | Yes             | Yes         | No         | No          | No           | No            | No              | Yes              | No                | No                 | No          |
| Uganda                                | Yes             | Yes         | No         | No          | No           | No            | Yes             | Yes              | No                | No                 | No          |
| Moldova, Republic of                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Djibouti                              | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Mongolia                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Egypt                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Netherlands                           | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Kyrgyzstan                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | No          |
| Czechia                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Botswana                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Hungary                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Timor-Leste                           | No              | Yes         | No         | No          | No           | No            | No              | No               | Yes               | No                 | No          |
| Puerto Rico                           | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Ireland                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Morocco                               | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Guatemala                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | No          |
| Lao People's Democratic Republic      | Yes             | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Panama                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Suriname                              | No              | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Myanmar                               | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Nepal                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| United Arab Emirates                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Germany                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Dominican Republic                    | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Fiji                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Tanzania, United Republic of          | Yes             | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Honduras                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Russian Federation                    | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Latvia                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Faroe Islands                         | No              | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Slovenia                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Trinidad and Tobago                   | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Venezuela, Bolivarian Republic of     | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Haiti                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Ukraine                               | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Dominica                              | No              | Yes         | No         | Yes         | No           | No            | No              | Yes              | Yes               | No                 | No          |
| Papua New Guinea                      | Yes             | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Albania                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Greece                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Sweden                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Hong Kong                             | Yes             | No          | Yes        | Yes         | Yes          | Yes           | No              | No               | No                | No                 | No          |
| Viet Nam                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Zambia                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Bahrain                               | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Angola                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Nicaragua                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Sierra Leone                          | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Slovakia                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Liberia                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Cambodia                              | Yes             | Yes         | No         | No          | No           | No            | No              | Yes              | Yes               | No                 | No          |
| Tajikistan                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Denmark                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Yemen                                 | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Kuwait                                | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Bahamas                               | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Zimbabwe                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Barbados                              | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Türkiye                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Aruba                                 | Yes             | No          | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Finland                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Norway                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Qatar                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Uzbekistan                            | No              | Yes         | No         | No          | No           | No            | No              | No               | Yes               | No                 | No          |
| Sri Lanka                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Luxembourg                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Niger                                 | Yes             | Yes         | No         | No          | No           | No            | No              | Yes              | No                | Yes                | No          |
| Martinique                            | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Réunion                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Chad                                  | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| France                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Singapore                             | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Bosnia and Herzegovina                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Costa Rica                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Brunei Darussalam                     | No              | Yes         | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Guinea                                | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Mali                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Mozambique                            | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| North Macedonia                       | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Kazakhstan                            | Yes             | Yes         | No         | No          | No           | No            | No              | Yes              | No                | No                 | No          |
| Virgin Islands, U.S.                  | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Burkina Faso                          | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Gambia                                | No              | Yes         | No         | No          | No           | No            | No              | Yes              | No                | Yes                | No          |
| Belarus                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Japan                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Mauritania                            | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Colombia                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Côte d'Ivoire                         | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Jordan                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| New Caledonia                         | No              | Yes         | No         | Yes         | No           | No            | No              | Yes              | Yes               | No                 | No          |
| Nigeria                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Iraq                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| United States                         | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Samoa                                 | No              | Yes         | No         | Yes         | No           | No            | No              | Yes              | Yes               | No                 | No          |
| Brazil                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Kenya                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Malta                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Libya                                 | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Lebanon                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Spain                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Australia                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Bhutan                                | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Cabo Verde                            | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Liechtenstein                         | Yes             | No          | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Italy                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Togo                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Central African Republic              | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Serbia                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Antigua and Barbuda                   | Yes             | No          | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Madagascar                            | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| United Kingdom                        | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| New Zealand                           | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Georgia                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Canada                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| India                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Congo, The Democratic Republic of the | No              | Yes         | No         | No          | No           | No            | No              | Yes              | No                | No                 | No          |
| Vanuatu                               | No              | Yes         | No         | Yes         | No           | No            | No              | Yes              | Yes               | No                 | No          |
| Guyana                                | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Belize                                | Yes             | No          | No         | No          | No           | No            | No              | No               | Yes               | Yes                | No          |
| Peru                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Israel                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Rwanda                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Chile                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Taiwan, Province of China             | Yes             | Yes         | Yes        | Yes         | Yes          | Yes           | No              | No               | No                | No                 | No          |
| Malawi                                | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Guinea-Bissau                         | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Belgium                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Mauritius                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Pakistan                              | Yes             | No          | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Mexico                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Argentina                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Gabon                                 | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Philippines                           | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Oman                                  | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Bulgaria                              | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| American Samoa                        | No              | Yes         | No         | Yes         | No           | No            | No              | Yes              | No                | No                 | No          |
| Jamaica                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Romania                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Uruguay                               | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Lithuania                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Indonesia                             | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Poland                                | Yes             | Yes         | No         | No          | No           | No            | No              | No               | No                | No                 | Yes         |
| Equatorial Guinea                     | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |
| Lesotho                               | No              | Yes         | No         | No          | No           | No            | No              | No               | No                | Yes                | No          |

In [7]:
json.dump(included, open(DATA_PATH / "config/included.json", "w"))